# Pretrain a small language model with TrainLM

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/main/notebooks/TrainLM_Native_TorchTPU_From_Scratch.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/Dhiraj309/TrainLM/main/notebooks/TrainLM_Native_TorchTPU_From_Scratch.ipynb)

| Goal | Settings |
| --- | --- |
| Scratch | `MODEL_INITIALIZATION="random"`, `DATA_FORMAT="text"` |
| Continued pretraining | `MODEL_INITIALIZATION="pretrained"`, `DATA_FORMAT="text"` |
| Pretokenized HF corpus | Either initialization; set `DATA_FORMAT="tokens"` and choose an HF Dataset with `input_ids` |

Set the model/tokenizer IDs, dataset ID/revision and splits in the training cell. The tokenized route pins the dataset revision and reloads the same splits on resume. Set `TOKENIZER_VOCAB_SHA256` to the digest supplied with your token corpus.

Choose a new `RUN_DIR` for each independent run. The default budget is a workflow smoke run, not a quality or performance result. For checkpoint continuation use the resume cell; for recovery on another machine opt in to the larger ZIP, which may contain private prepared data. Publishing is optional and private by default. This recipe targets one native TorchTPU device; the links do not provision an accelerator.

See the [pretraining guide](../README.md#start-here-pretrain-a-language-model-from-scratch) for configuration details and limits.


In [ ]:
import subprocess
import sys

source = "trainlm @ git+https://github.com/Dhiraj309/TrainLM.git@main"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", source],
    check=True,
)
requirements = [
    "huggingface-hub>=1.5,<2",
    "PyYAML>=6,<7",
    "transformers>=5.15,<6",
    "datasets>=3,<5",
    "tokenizers>=0.21,<1",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *requirements],
    check=True,
)
print("TrainLM installed without asking pip to replace PyTorch or TorchTPU.")
from importlib.metadata import PackageNotFoundError, version

for package_name in ("torch-tpu", "libtpu"):
    try:
        print(f"{package_name}=={version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name}=MISSING; native TPU needs a runtime that provides TorchTPU")


In [ ]:
import json
from importlib.metadata import distribution
from trainlm import TrainLMPretrainingConfig, run_pretraining
source = json.loads(distribution("trainlm").read_text("direct_url.json") or "{}")
source_revision = source.get("vcs_info", {}).get("commit_id")
assert source_revision, "Reinstall TrainLM from Git to record its exact commit."
RUN_DIR = "trainlm-pretraining-fresh"
FRESH_MAX_TOKENS, FRESH_MAX_TRAIN_EXAMPLES, LOGGING_STEPS = 10_000, 2_000, 10
MODEL_ID, MODEL_INITIALIZATION = "openai-community/gpt2", "random"
TOKENIZER_ID = MODEL_ID
MODEL_REVISION = None
TOKENIZER_REVISION = MODEL_REVISION
DATASET_ID, DATASET_CONFIG_NAME, DATASET_REVISION = "roneneldan/TinyStories", None, "main"
TRAIN_SPLIT, VALIDATION_SPLIT = "train", "validation"
DATA_FORMAT = "text"  # "tokens" for HF Dataset rows with input_ids.
TOKENIZER_VOCAB_SHA256 = None  # For tokens, set the digest supplied with that corpus.
run_inputs = {}
if DATA_FORMAT == "tokens":
    from datasets import load_dataset
    from huggingface_hub import HfApi
    from transformers import AutoTokenizer
    from trainlm import tokenizer_vocab_sha256

    # Pin the Dataset once so a later resume reads the same immutable version.
    DATASET_REVISION = HfApi().dataset_info(
        DATASET_ID, revision=DATASET_REVISION
    ).sha
    train_dataset = load_dataset(
        DATASET_ID, name=DATASET_CONFIG_NAME, revision=DATASET_REVISION,
        split=TRAIN_SPLIT,
    )
    eval_dataset = load_dataset(
        DATASET_ID, name=DATASET_CONFIG_NAME, revision=DATASET_REVISION,
        split=VALIDATION_SPLIT,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        TOKENIZER_ID, revision=TOKENIZER_REVISION
    )
    actual_vocab_sha256 = tokenizer_vocab_sha256(tokenizer)
    print(f"tokenizer_vocab_sha256={actual_vocab_sha256}")
    if TOKENIZER_VOCAB_SHA256 is not None and TOKENIZER_VOCAB_SHA256 != actual_vocab_sha256:
        raise ValueError("The corpus tokenizer digest does not match TOKENIZER_ID.")
    run_inputs = {"train_dataset": train_dataset, "eval_dataset": eval_dataset}
    dataset_source = "provided"
else:
    dataset_source = "hub"
config = TrainLMPretrainingConfig(
    output_dir=RUN_DIR,
    max_tokens=FRESH_MAX_TOKENS, max_train_examples=FRESH_MAX_TRAIN_EXAMPLES,
    sequence_length=128, logging_steps=LOGGING_STEPS,
    model_name_or_path=MODEL_ID, model_revision=MODEL_REVISION,
    model_initialization=MODEL_INITIALIZATION,
    tokenizer_name_or_path=TOKENIZER_ID, tokenizer_revision=TOKENIZER_REVISION,
    dataset_source=dataset_source, dataset_id=DATASET_ID,
    dataset_config_name=DATASET_CONFIG_NAME, dataset_revision=DATASET_REVISION,
    train_split=TRAIN_SPLIT, validation_split=VALIDATION_SPLIT,
    data_format=DATA_FORMAT, text_field="text", accelerator="native_tpu",
    expected_tokenizer_vocab_sha256=TOKENIZER_VOCAB_SHA256,
)
print(json.dumps(config.training_plan(inspect_model=True), indent=2))
result = run_pretraining(
    config, **run_inputs, show_progress=True, source_revision=source_revision
)
print(result.summary())
print(f"checkpoint: {result.checkpoint_path}")
print(f"report: {result.report_path}")
print(f"model: {result.export_path}")

In [ ]:
import json
from importlib.metadata import distribution
from dataclasses import replace
from trainlm import restore_pretraining_report, run_pretraining
source = json.loads(distribution("trainlm").read_text("direct_url.json"))
source_revision = source["vcs_info"]["commit_id"]
RUN_DIR, RESUME_MODE = "trainlm-pretraining-fresh", "skip"  # "resume" or "extend"
if RESUME_MODE in {"resume", "extend"}:
    saved = restore_pretraining_report(f"{RUN_DIR}/report.json")
    resume_config = saved.config
    extend_budget = RESUME_MODE == "extend"
    if extend_budget:
        extra_tokens = max(2_000, resume_config.max_tokens // 5)
        resume_config = replace(resume_config, max_tokens=resume_config.max_tokens + extra_tokens)
    options = {"resume_from_checkpoint": saved.checkpoint_path, "extend_training_budget": extend_budget}
    resume_inputs = {}
    if resume_config.dataset_source == "provided":
        from datasets import load_dataset
        train_dataset = load_dataset(
            resume_config.dataset_id, name=resume_config.dataset_config_name,
            revision=resume_config.dataset_revision, split=resume_config.train_split,
        )
        eval_dataset = load_dataset(
            resume_config.dataset_id, name=resume_config.dataset_config_name,
            revision=resume_config.dataset_revision, split=resume_config.validation_split,
        )
        resume_inputs = {"train_dataset": train_dataset, "eval_dataset": eval_dataset}
    print(json.dumps(resume_config.training_plan(**options), indent=2))
    resumed = run_pretraining(
        resume_config, **options, **resume_inputs, show_progress=True,
        source_revision=source_revision,
        report_path=f"{RUN_DIR}/resume-report.json", export_path=f"{RUN_DIR}/resumed-export",
    )
    print(resumed.summary())
    print(f"Resumed from: {saved.checkpoint_path} | report: {resumed.report_path} | model: {resumed.export_path}")
else:
    print("Choose RESUME_MODE='resume' or 'extend', then run this cell only.")

## Verify the exported model with Transformers

This reload uses the ordinary Transformers API on CPU; the exported model does not need TrainLM for inference.


In [ ]:
RUN_RESUME = globals().get("RESUME_MODE", "skip") in {"resume", "extend"}
run = resumed if RUN_RESUME else result
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_path = str(run.export_path)
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True)
prompt = tokenizer("Once upon a time", return_tensors="pt")
with torch.inference_mode():
    generated = model.generate(**prompt, max_new_tokens=24, do_sample=False)
print(tokenizer.decode(generated[0], skip_special_tokens=True))


In [ ]:
from trainlm import bundle_pretraining_artifacts
RUN_RESUME = globals().get("RESUME_MODE", "skip") in {"resume", "extend"}
run = resumed if RUN_RESUME else result
INCLUDE_RESUME_STATE = False  # True adds checkpoints and prepared data; archive may be large/private.
if RUN_RESUME:
    bundle = bundle_pretraining_artifacts(
        resumed.report_path, include_resume_state=INCLUDE_RESUME_STATE
    )
else:
    bundle = bundle_pretraining_artifacts(
        result.report_path, include_resume_state=INCLUDE_RESUME_STATE
    )
try:
    from google.colab import files
except ImportError:
    print(f"Artifact bundle: {bundle}")
else:
    files.download(str(bundle))


## Optional: publish the completed model privately

Set `PUBLISH_TO_HUB=True` only after reviewing the model and data licenses. TrainLM creates new repositories as private by default; the upload excludes the run report, prepared data, and checkpoints. Authenticate with `hf auth login` or `HF_TOKEN` before running this cell.


In [ ]:
PUBLISH_TO_HUB = False
HUB_REPO_ID = "your-account/your-model"
if PUBLISH_TO_HUB:
    receipt = run.push_to_hub(HUB_REPO_ID)
    print(f"Published: {receipt.repo_url} at {receipt.revision}")
else:
    print("Publishing is disabled.")
